<a href="https://colab.research.google.com/github/ElifBerra/crewai-multiagent-rag-lab/blob/main/CrewAI_MultiAgent_Orchestration_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install "crewai[litellm]" --default-timeout=100

  Using cached crewai-1.15.15-py3-none-any.whl.metadata (37 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached aiosqlite-0.21.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached cel_python-0.5.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached chromadb-1.1.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached crewai_cli-1.15.15-py3-none-any.whl.metadata (1.7 kB)
  Using cached crewai_core-1.15.15-py3-none-any.whl.metadata (1.2 kB)
  Using cached instructor-1.15.4-py3-none-any.whl.metadata (12 kB)
  Using cached json_repair-0.60.1-py3-none-any.whl.metadata (19 kB)
  Using cached json5-0.10.0-py3-none-any.whl.metadata (34 kB)
  Using cached jsonref-1.1.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached lancedb-0.30.0-cp39-abi3-manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached mcp-1.28.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached open

In [5]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00


In [9]:
import os
import asyncio
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

# 1. Groq API Anahtarını Çek
groq_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_key

# 2. Groq'u OpenAI-compatible Endpoint olarak Tanımlıyoruz
# Bu sayede CrewAI araya Anthropic/OpenAI caching parametrelerini eklemez
groq_llm = LLM(
    model="openai/llama-3.1-8b-instant",
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.7
)

# 3. Ajanlar
arastirmaci = Agent(
    role='Kıdemli Araştırmacı',
    goal='{konu} hakkında doğru, güncel ve önemli bilgileri toplamak',
    backstory='Karmaşık konuları netleştiren, titiz bir araştırmacısın.',
    verbose=True,
    llm=groq_llm
)

yazar = Agent(
    role='Teknik İçerik Yazarı',
    goal='Araştırma notlarını akıcı, anlaşılır bir yazıya dönüştürmek',
    backstory='Teknik konuları herkesin anlayacağı dile çeviren bir yazarsın.',
    verbose=True,
    llm=groq_llm
)

# 4. Görevler
arastirma_gorevi = Task(
    description='{konu} konusunu araştır. En önemli 5 noktayı belirle.',
    expected_output='Madde madde, her biri 1-2 cümlelik 5 kilit bulgu.',
    agent=arastirmaci,
)

yazma_gorevi = Task(
    description='Araştırma bulgularını kullanarak kısa bir blog yazısı yaz.',
    expected_output='Giriş-gelişme-sonuç yapısında, ~300 kelimelik bir yazı.',
    agent=yazar,
    context=[arastirma_gorevi],
)

# 5. Ekip
ekip = Crew(
    agents=[arastirmaci, yazar],
    tasks=[arastirma_gorevi, yazma_gorevi],
    process=Process.sequential,
    verbose=True
)

# 6. Asenkron Çalıştırma
async def calistir():
    sonuc = await ekip.kickoff_async(inputs={'konu': 'RAG sistemlerinde chunking'})
    print("\n" + "="*50)
    print("📌 NİHAİ ÇIKTI:")
    print("="*50)
    print(sonuc)

await calistir()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7c4c963d-7d82-4e98-aa36-fcff1c1a10d7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: RAG sistemlerinde chunking konusunu araştır. En önemli 5 noktayı belirle.                                │
│  ID: bccf9fa7-4b29-4374-91eb-b19ed8fa34b4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kıdemli Araştırmacı                                                                                     │
│                                                                                                                 │
│  Task: RAG sistemlerinde chunking konusunu araştır. En önemli 5 noktayı belirle.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Kıdemli Araştırmacı                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  RAG sistemlerinde chunking konusunu araştırdım ve aşağıdaki 5 önemli nokta belirledim:                         │
│                                                                                                                 │
│  1. **Chunking Tanımı ve Önemi**: Chunking, RAG sistemlerinde verilerin daha küçük ve işlenebilir bloklara      │
│  (chunk) ayırılmasına verilen addır. Bu proces, verilerin daha kolay işlenebilir olması, hata oranının          │
│  azaltılması ve sistemlerin daha etkili bir şekilde çalışması için önemlidir.                                   │
│                                                                                                                 │
│  2. **Chunking Türleri**: RAG sistemlerinde chunking iki ana türde gerçekleşir: otomatik chunking ve manuel     │
│  chunking. Otomatik chunking, sistemlerin kendi kendine verilerin chunk'ları oluşturmasıdır, manuel chunking    │
│  ise insan müdahalesiyle verilerin chunk'ları oluşturulmasıdır.                                                 │
│                                                                                                                 │
│  3. **Chunking Algoritmaları**: RAG sistemlerinde chunking için kullanılan algoritmalar, verilerin niteliğine   │
│  ve sistemin gereksinimlerine göre değişir. Örneğin, bazı sistemler verilerin içerik analizini kullanarak       │
│  chunk'ları oluştururken, diğer sistemler ise verilerin yapısal özelliklerine göre chunk'ları oluşturabilir.    │
│                                                                                                                 │
│  4. **Chunkingin RAG Sistemlerinde Önemi**: Chunking, RAG sistemlerinin etkinlik ve verimliliğini artırmada     │
│  önemli bir rol oynar. Verilerin daha küçük bloklara ayırılması, sistemin daha hızlı ve daha doğru bir şekilde  │
│  verilere erişmesine olanak tanır. Ayrıca, chunking verilerin daha kolay işlenebilir olması nedeniyle hata      │
│  oranının azaltılmasına yardımcı olur.                                                                          │
│                                                                                                                 │
│  5. **Chunking Uygulamaları**: RAG sistemlerinde chunking, çeşitli alanlarda kullanılmaktadır. Örneğin, doğal   │
│  dil işleme, veri madenciliği, veri depolama ve veri analizinde chunking önemli bir role sahiptir. Chunking,    │
│  sistemin verilere daha hızlı ve daha doğru bir şekilde erişmesine olanak tanır ve verilerin daha kolay         │
│  işlenebilir olması nedeniyle hata oranının azaltılmasına yardımcı olur.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Teknik İçerik Yazarı                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **RAG Sistemlerinde Chunking: Verilerin İşlenebilirliğini Artıran Önemli Bir Proses**                          │
│                                                                                                                 │
│  RAG (Referans ve Ayrıntı Görüntüleme) sistemleri, büyük miktarda veriyi işleyerek anlamlı sonuçlar elde        │
│  etmemizi sağlar. Bu sistemler, verilerin işlenebilirliğini artırmak için çeşitli teknikler kullanır ve         │
│  bunlardan biri de chunking'dir. Chunking, verilerin daha küçük ve işlenebilir bloklara ayırılmasına verilen    │
│  addır. Bu yazımızda, RAG sistemlerinde chunking konusunu araştırdık ve önemini anlamak için beş önemli         │
│  noktayı gözden geçireceğiz.                                                                                    │
│                                                                                                                 │
│  **Chunking Tanımı ve Önemi**                                                                                   │
│                                                                                                                 │
│  Chunking, RAG sistemlerinde verilerin işlenebilirliğini artırmak için kullanılan bir tekniktir. Verilerin      │
│  daha küçük bloklara ayırılması, sistemin daha hızlı ve doğru bir şekilde verilere erişmesine olanak tanır.     │
│  Ayrıca, chunking verilerin daha kolay işlenebilir olması nedeniyle hata oranının azaltılmasına yardımcı olur.  │
│  Bu nedenle, chunking RAG sistemlerinin etkinlik ve verimliliğini artırmada önemli bir role sahiptir.           │
│                                                                                                                 │
│  **Chunking Türleri**                                                                                           │
│                                                                                                                 │
│  RAG sistemlerinde chunking iki ana türde gerçekleşir: otomatik chunking ve manuel chunking. Otomatik           │
│  chunking, sistemlerin kendi kendine verilerin chunk'ları oluşturmasıdır. Bu teknik, sistemlerin otomatik       │
│  olarak verilerin işlenebilirliğini artırmalarına olanak tanır. Manuel chunking ise insan müdahalesiyle         │
│  verilerin chunk'ları oluşturulmasıdır. Bu teknik, sistemin daha doğru ve anlamlı sonuçlar elde etmesine        │
│  yardımcı olur.                                                                                                 │
│                                                                                                                 │
│  **Chunking Algoritmaları**                                                                                     │
│                                                                                                                 │
│  RAG sistemlerinde chunking için kullanılan algoritmalar, verilerin niteliğine ve sistemin gereksinimlerine     │
│  göre değişir. Örneğin, bazı sistemler verilerin içerik analizini kullanarak chunk'ları oluştururken, diğer     │
│  sistemler ise verilerin yapısal özelliklerine göre chunk'ları oluşturabilir. Bu nedenle, sistemlerin seçeceği  │
│  algoritma, verilerin işlenebilirliğini artırmak için önemlidir.                                                │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Araştırma bulgularını kullanarak kısa bir blog yazısı yaz.                                               │
│  Agent: Teknik İçerik Yazarı                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📌 NİHAİ ÇIKTI:
**RAG Sistemlerinde Chunking: Verilerin İşlenebilirliğini Artıran Önemli Bir Proses**

RAG (Referans ve Ayrıntı Görüntüleme) sistemleri, büyük miktarda veriyi işleyerek anlamlı sonuçlar elde etmemizi sağlar. Bu sistemler, verilerin işlenebilirliğini artırmak için çeşitli teknikler kullanır ve bunlardan biri de chunking'dir. Chunking, verilerin daha küçük ve işlenebilir bloklara ayırılmasına verilen addır. Bu yazımızda, RAG sistemlerinde chunking konusunu araştırdık ve önemini anlamak için beş önemli noktayı gözden geçireceğiz.

**Chunking Tanımı ve Önemi**

Chunking, RAG sistemlerinde verilerin işlenebilirliğini artırmak için kullanılan bir tekniktir. Verilerin daha küçük bloklara ayırılması, sistemin daha hızlı ve doğru bir şekilde verilere erişmesine olanak tanır. Ayrıca, chunking verilerin daha kolay işlenebilir olması nedeniyle hata oranının azaltılmasına yardımcı olur. Bu nedenle, chunking RAG sistemlerinin etkinlik ve verimliliğini artırmada önemli bir role sahipt

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7c4c963d-7d82-4e98-aa36-fcff1c1a10d7                                                                       │
│  Final Output: **RAG Sistemlerinde Chunking: Verilerin İşlenebilirliğini Artıran Önemli Bir Proses**            │
│                                                                                                                 │
│  RAG (Referans ve Ayrıntı Görüntüleme) sistemleri, büyük miktarda veriyi işleyerek anlamlı sonuçlar elde        │
│  etmemizi sağlar. Bu sistemler, verilerin işlenebilirliğini artırmak için çeşitli teknikler kullanır ve         │
│  bunlardan biri de chunking'dir. Chunking, verilerin daha küçük ve işlenebilir bloklara ayırılmasına verilen    │
│  addır. Bu yazımızda, RAG sistemlerinde chunking konusunu araştırdık ve önemini anlamak için beş önemli         │
│  noktayı gözden geçireceğiz.                                                                                    │
│                                                                                                                 │
│  **Chunking Tanımı ve Önemi**                                                                                   │
│                                                                                                                 │
│  Chunking, RAG sistemlerinde verilerin işlenebilirliğini artırmak için kullanılan bir tekniktir. Verilerin      │
│  daha küçük bloklara ayırılması, sistemin daha hızlı ve doğru bir şekilde verilere erişmesine olanak tanır.     │
│  Ayrıca, chunking verilerin daha kolay işlenebilir olması nedeniyle hata oranının azaltılmasına yardımcı olur.  │
│  Bu nedenle, chunking RAG sistemlerinin etkinlik ve verimliliğini artırmada önemli bir role sahiptir.           │
│                                                                                                                 │
│  **Chunking Türleri**                                                                                           │
│                                                                                                                 │
│  RAG sistemlerinde chunking iki ana türde gerçekleşir: otomatik chunking ve manuel chunking. Otomatik           │
│  chunking, sistemlerin kendi kendine verilerin chunk'ları oluşturmasıdır. Bu teknik, sistemlerin otomatik       │
│  olarak verilerin işlenebilirliğini artırmalarına olanak tanır. Manuel chunking ise insan müdahalesiyle         │
│  verilerin chunk'ları oluşturulmasıdır. Bu teknik, sistemin daha doğru ve anlamlı sonuçlar elde etmesine        │
│  yardımcı olur.                                                                                                 │
│                                                                                                                 │
│  **Chunking Algoritmaları**                                                                                     │
│                                                                                                                 │
│  RAG sistemlerinde chunking için kullanılan algoritmalar, verilerin niteliğine ve sistemin gereksinimlerine     │
│  göre değişir. Örneğin, bazı sistemler verilerin içerik analizini kullanarak chunk'ları oluştururken, diğer     │
│  sistemler ise verilerin yapısal özelliklerine göre chunk'ları oluşturabilir. Bu nedenle, sistemlerin seçeceği  │
│  algoritma, verilerin işlenebilirliğini artırmak için önemlidir.                                                │
│                                                       

In [12]:
# 1. Editör Agent
editor = Agent(
    role='Baş Editör',
    goal='Yazarın metnini dil bilgisi, akıcılık ve okunabilirlik açısından mükemmelleştirmek',
    backstory='Yayına giren her yazının en yüksek kalitede olmasını sağlayan titiz bir baş editörsün.',
    verbose=True,
    llm=groq_llm
)

# 2. Editör Görevi (Düzeltilmiş Tırnak Yapısı)
editor_gorevi = Task(
    description="Teknik İçerik Yazarı'nın hazırladığı blog yazısını incele. Dil bilgisi hatalarını düzelt, akıcılığı artır ve yayınlanmaya hazır nihai hali sun.",
    expected_output='Profesyonelce düzenlenmiş, imla hatalarından arındırılmış, başlıkları ve vurguları düzenlenmiş nihai blog yazısı.',
    agent=editor,
    context=[yazma_gorevi]  # Yazarın çıktısını girdi alır!
)

In [14]:
!pip install crewai-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.13.0
    Uninstalling tiktoken-0.13.0:
      Successfully uninstalled tiktoken-0.13.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavio

In [3]:
import os
from google.colab import userdata
from crewai import Agent, LLM
from crewai_tools import SerperDevTool

# 1. API Anahtarını al ve LLM'i bu hücreye özel tanımla
groq_key = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = groq_key

groq_llm = LLM(
    model="openai/llama-3.1-8b-instant",
    api_key=groq_key,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.7
)

# 2. Arama aracını başlatıyoruz
arama_araci = SerperDevTool()

# 3. Araştırmacı Agent'a bu aracı bağlayarak oluşturuyoruz
arastirmaci = Agent(
    role='Kıdemli Araştırmacı',
    goal='{konu} hakkında doğru ve güncel bilgileri web üzerinden toplamak',
    backstory='İnternetteki en güncel kaynakları tarayıp analiz eden titiz bir araştırmacısın.',
    verbose=True,
    tools=[arama_araci],  # <-- Araç bağlama noktası
    llm=groq_llm
)

print("✅ Agent ve Tool başarıyla tanımlandı!")

✅ Agent ve Tool başarıyla tanımlandı!
